In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator, NullFormatter, AutoMinorLocator
from matplotlib.colors import Normalize
from matplotlib.colors import to_rgba
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import LogLocator, NullFormatter

from sklearn.isotonic import IsotonicRegression
from scipy.ndimage import gaussian_filter

import sys
import os
sys.path.append(os.getcwd())
sys.path.append('/home/habjan.e/TNG/cluster_deprojection/conditional_diffusion_model')

import matplotlib as mpl
mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["TeX Gyre Pagella", "Book Antiqua", "Palatino Linotype", "DejaVu Serif"]
})

import pickle
import jax
import jax.numpy as jnp

from pathlib import Path
import re

from conditional_diffusion_3d_model import ConditionalUNet3D, DiffusionModelConfig
from train_conditional_diffusion import preload_hdf5_to_memory, train_model

### Simulation colors

In [ ]:
# Okabe–Ito color-blind-friendly colors

tng300_2_color = "#0072B2"
tng300_3_color = "#D55E00"

### Import TNG specifications

In [ ]:
baseUrl = 'http://www.tng-project.org/api/'
dirc_path = '/home/habjan.e/'

sys.path.append(dirc_path + 'TNG/Codes/TNG_workshop')
import iapi_TNG as iapi

dirc=dirc_path + 'TNG/TNG_workshop/'
r=iapi.get(baseUrl)
TNG_data_path = dirc_path + 'TNG/Data/'

### Import $M_{200}$ for each simulation

In [ ]:
num_tng_clusters = 100

sim = 'TNG300-2'
simUrl = baseUrl+sim
simdata = iapi.get(simUrl)
Group_M_Crit200_2 = iapi.getHaloField(field = 'Group_M_Crit200', simulation=sim, snapshot=99, fileName= TNG_data_path+'TNG_data/'+sim+'_Group_M_Crit200', rewriteFile=0)
M_Crit200_TNG2 = Group_M_Crit200_2[:num_tng_clusters]

sim = 'TNG300-3'
simUrl = baseUrl+sim
simdata = iapi.get(simUrl)
Group_M_Crit200_3 = iapi.getHaloField(field = 'Group_M_Crit200', simulation=sim, snapshot=99, fileName= TNG_data_path+'TNG_data/'+sim+'_Group_M_Crit200', rewriteFile=0)
M_Crit200_TNG3 = Group_M_Crit200_3[:num_tng_clusters]

### Figure 1: $M_{200}$ distribution by sim

In [ ]:
mass_unit_conversion = 1e10 / simdata["hubble"]
bins = np.linspace(13.55, 15.45, 30)

# Remove invalid values before taking the logarithm
mask_2 = np.isfinite(M_Crit200_TNG2) & (M_Crit200_TNG2 > 0)
mask_3 = np.isfinite(M_Crit200_TNG3) & (M_Crit200_TNG3 > 0)

log_mass_2 = np.log10(
    M_Crit200_TNG2[mask_2] * mass_unit_conversion
)
log_mass_3 = np.log10(
    M_Crit200_TNG3[mask_3] * mass_unit_conversion
)

with plt.rc_context({
    "font.size": 10,
    "axes.labelsize": 11,
    "axes.linewidth": 0.8,
    "legend.fontsize": 9,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "xtick.major.size": 5,
    "ytick.major.size": 5,
    "xtick.minor.size": 3,
    "ytick.minor.size": 3,
}):
    fig, ax = plt.subplots(
        figsize=(5.2, 4.2),
        constrained_layout=True,
    )

    ax.hist(
        log_mass_2,
        bins=bins,
        #histtype="step",
        alpha = 0.75,
        linewidth=2.0,
        linestyle="-",
        color=tng300_2_color,
        label="TNG300-2",
    )

    ax.hist(
        log_mass_3,
        bins=bins,
        #histtype="step",
        alpha = 0.75,
        linewidth=2.0,
        linestyle="--",
        color=tng300_3_color,
        label="TNG300-3",
    )

    ax.set_xlabel(
        r"$\log_{10}\!\left(M_{200}/M_\odot\right)$"
    )
    ax.set_ylabel("Number of Haloes")
    ax.set_xlim(bins[0], bins[-1])

    ax.xaxis.set_minor_locator(AutoMinorLocator())
    ax.yaxis.set_minor_locator(AutoMinorLocator())

    ax.grid(
        #axis="y",
        color="0.88",
        linewidth=0.6,
        zorder=0,
    )
    ax.legend(frameon=False)

    fig.savefig(
        "halo_mass_distribution.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()

# Figure 2: G-band versus stellar mass

### Import g-band and stellar mass information

In [ ]:
sim = 'TNG300-2'
simUrl = baseUrl+sim
simdata = iapi.get(simUrl)
SubhaloMassType_2 = iapi.getSubhaloField('SubhaloMassType', simulation=sim, snapshot=99, fileName=TNG_data_path+'TNG_data/'+sim+'_SubhaloMassType', rewriteFile=0)
SubhaloStellarPhotometrics_2 = iapi.getSubhaloField('SubhaloStellarPhotometrics', simulation=sim, snapshot=99, fileName=TNG_data_path+'TNG_data/'+sim+'_SubhaloStellarPhotometrics', rewriteFile=0)
StellarMass_2 = SubhaloMassType_2[:, 4]
GBand_2 = SubhaloStellarPhotometrics_2[:, 4]


sim = 'TNG300-3'
simUrl = baseUrl+sim
simdata = iapi.get(simUrl)
SubhaloMassType_3 = iapi.getSubhaloField('SubhaloMassType', simulation=sim, snapshot=99, fileName=TNG_data_path+'TNG_data/'+sim+'_SubhaloMassType', rewriteFile=0)
SubhaloStellarPhotometrics_3 = iapi.getSubhaloField('SubhaloStellarPhotometrics', simulation=sim, snapshot=99, fileName=TNG_data_path+'TNG_data/'+sim+'_SubhaloStellarPhotometrics', rewriteFile=0)
StellarMass_3 = SubhaloMassType_3[:, 4]
GBand_3 = SubhaloStellarPhotometrics_3[:, 4]

tng_stellar_mass = np.concatenate([StellarMass_2, StellarMass_3])
tng_gband = np.concatenate([GBand_2, GBand_3])

### Function for removing subhalos

In [ ]:
def mag_g_proxy_mask(
    mstar_cal,
    mag_g_cal,
    mstar_target,
    mag_cut=-18.0,
    logmass_bins=None,
    min_bin_count=30,
    method="stochastic",
    completeness_level=0.90,
    seed=None,
    return_info=False,
):
    """
    Select subhalos in a simulation without g-band magnitudes using a
    selection function calibrated from a simulation with magnitudes.

    Parameters
    ----------
    mstar_cal : array-like
        Stellar masses in the calibration simulation.
    mag_g_cal : array-like
        Corresponding g-band absolute magnitudes.
    mstar_target : array-like
        Stellar masses in the simulation to which the selection is applied.
    mag_cut : float, default=-18
        Magnitude threshold. Objects with mag_g <= mag_cut pass.
    logmass_bins : array-like, optional
        Bin edges in log10(stellar mass). By default, 0.15-dex bins are used.
    min_bin_count : int, default=30
        Minimum number of calibration objects required in a mass bin.
    method : {"stochastic", "hard"}
        "stochastic": retain each target object with probability
        P(mag_g <= mag_cut | Mstar).

        "hard": retain all target objects above the stellar mass where the
        fitted selection probability reaches `completeness_level`.
    completeness_level : float, default=0.90
        Completeness used when method="hard".
    seed : int, optional
        Random seed for reproducible stochastic masks.
    return_info : bool, default=False
        If True, also return selection probabilities and calibration details.

    Returns
    -------
    mask : ndarray of bool
        Boolean selection mask for `mstar_target`.

    info : dict, optional
        Returned only when `return_info=True`.
    """
    mstar_cal = np.asarray(mstar_cal, dtype=float)
    mag_g_cal = np.asarray(mag_g_cal, dtype=float)
    mstar_target = np.asarray(mstar_target, dtype=float)

    if mstar_cal.shape != mag_g_cal.shape:
        raise ValueError("mstar_cal and mag_g_cal must have the same shape.")

    if method not in {"stochastic", "hard"}:
        raise ValueError("method must be 'stochastic' or 'hard'.")

    if not 0 < completeness_level <= 1:
        raise ValueError("completeness_level must be between 0 and 1.")

    # Remove invalid calibration entries.
    valid_cal = (
        np.isfinite(mstar_cal)
        & np.isfinite(mag_g_cal)
        & (mstar_cal > 0)
    )

    logm_cal = np.log10(mstar_cal[valid_cal])
    passes_mag_cut = mag_g_cal[valid_cal] <= mag_cut

    if logmass_bins is None:
        lower = np.floor(logm_cal.min() / 0.15) * 0.15
        upper = np.ceil(logm_cal.max() / 0.15) * 0.15 + 0.15
        logmass_bins = np.arange(lower, upper, 0.15)
    else:
        logmass_bins = np.asarray(logmass_bins, dtype=float)

    bin_centers = 0.5 * (logmass_bins[:-1] + logmass_bins[1:])

    n_all, _ = np.histogram(logm_cal, bins=logmass_bins)
    n_pass, _ = np.histogram(logm_cal[passes_mag_cut], bins=logmass_bins)

    raw_completeness = np.divide(
        n_pass,
        n_all,
        out=np.full(n_all.shape, np.nan, dtype=float),
        where=n_all > 0,
    )

    usable = (n_all >= min_bin_count) & np.isfinite(raw_completeness)

    if usable.sum() < 2:
        raise ValueError(
            "Too few populated stellar-mass bins to calibrate the selection."
        )

    calibration_mass = bin_centers[usable]

    # Enforce the physically expected non-decreasing completeness with mass.
    isotonic = IsotonicRegression(
        increasing=True,
        y_min=0.0,
        y_max=1.0,
        out_of_bounds="clip",
    )

    fitted_completeness = isotonic.fit_transform(
        calibration_mass,
        raw_completeness[usable],
        sample_weight=n_all[usable],
    )

    # Calculate selection probabilities for the target simulation.
    target_valid = np.isfinite(mstar_target) & (mstar_target > 0)
    probabilities = np.zeros(mstar_target.shape, dtype=float)

    probabilities[target_valid] = np.interp(
        np.log10(mstar_target[target_valid]),
        calibration_mass,
        fitted_completeness,
        left=0.0,
        right=1.0,
    )

    mass_threshold = None

    if method == "stochastic":
        rng = np.random.default_rng(seed)
        mask = target_valid & (
            rng.random(mstar_target.shape) < probabilities
        )

    else:
        reaches_level = fitted_completeness >= completeness_level

        if not np.any(reaches_level):
            raise ValueError(
                f"The fitted selection never reaches "
                f"{completeness_level:.0%} completeness."
            )

        first = np.flatnonzero(reaches_level)[0]

        # Interpolate the mass at which completeness crosses the requested level.
        if first == 0:
            logmass_threshold = calibration_mass[0]
        else:
            logmass_threshold = np.interp(
                completeness_level,
                fitted_completeness[first - 1:first + 1],
                calibration_mass[first - 1:first + 1],
            )

        mass_threshold = 10**logmass_threshold
        mask = target_valid & (mstar_target >= mass_threshold)

    if return_info:
        info = {
            "probability": probabilities,
            "bin_centers_log10": bin_centers,
            "n_all": n_all,
            "n_pass": n_pass,
            "raw_completeness": raw_completeness,
            "calibration_logmass": calibration_mass,
            "fitted_completeness": fitted_completeness,
            "mass_threshold": mass_threshold,
        }
        return mask, info

    return mask

### Run the code

In [ ]:
stellar_mass_mask, cut_info = mag_g_proxy_mask(
    mstar_cal=tng_stellar_mass,
    mag_g_cal=tng_gband,
    mstar_target=tng_stellar_mass,
    mag_cut=-18,
    method="stochastic",
    seed=42,
    return_info=True,
)

### Make the plot

In [ ]:
mass_unit_conversion = 1e10 / simdata["hubble"]
mag_cut = -18.0

mass = tng_stellar_mass * mass_unit_conversion
stellar_mass_mask = np.asarray(stellar_mass_mask, dtype=bool)

mask_no_cut = (
    (mass > 0)
    & np.isfinite(mass)
    & np.isfinite(tng_gband)
)
mask_cut = mask_no_cut & stellar_mass_mask

logmass_all = np.log10(mass[mask_no_cut])
magnitude_all = tng_gband[mask_no_cut]

logmass_cut = np.log10(mass[mask_cut])
magnitude_cut = tng_gband[mask_cut]


# Shared bins for both distributions
logmass_edges = np.linspace(logmass_all.min(), logmass_all.max(), 60)
magnitude_edges = np.linspace(
    magnitude_all.min(),
    magnitude_all.max(),
    60,
)

logmass_centers = 0.5 * (
    logmass_edges[:-1] + logmass_edges[1:]
)
magnitude_centers = 0.5 * (
    magnitude_edges[:-1] + magnitude_edges[1:]
)

x_centers = 10**logmass_centers


def normalized_histogram(logmass, magnitude):
    """Create a normalized and mildly smoothed 2D histogram."""
    histogram, _, _ = np.histogram2d(
        logmass,
        magnitude,
        bins=(logmass_edges, magnitude_edges),
    )

    histogram = gaussian_filter(
        histogram.astype(float),
        sigma=1.0,
    )

    return histogram / histogram.sum()


def enclosed_density_level(histogram, fraction):
    """Find the density threshold enclosing a given fraction."""
    density = np.sort(histogram[histogram > 0].ravel())[::-1]

    cumulative = np.cumsum(density)
    cumulative /= cumulative[-1]

    index = np.searchsorted(cumulative, fraction)
    index = min(index, density.size - 1)

    return density[index]


hist_all = normalized_histogram(
    logmass_all,
    magnitude_all,
)
hist_cut = normalized_histogram(
    logmass_cut,
    magnitude_cut,
)

fractions = (0.50, 0.80, 0.95)

fill_alphas = {
    0.50: 0.50,
    0.80: 0.18,
    0.95: 0.08,
}

line_alphas = {
    0.50: 0.95,
    0.80: 0.65,
    0.95: 0.35,
}

colors = {
    "all": "#E57373",
    "cut": "#009E73",
}


def draw_filled_density(ax, histogram, color):
    """Draw filled 50%, 80%, and 95% density regions."""
    thresholds = {
        fraction: enclosed_density_level(histogram, fraction)
        for fraction in fractions
    }

    # Density thresholds increase inward:
    # 95% boundary < 80% boundary < 50% boundary.
    fill_levels = [
        thresholds[0.95],
        thresholds[0.80],
        thresholds[0.50],
        np.nextafter(histogram.max(), np.inf),
    ]

    fill_colors = [
        to_rgba(color, fill_alphas[0.95]),
        to_rgba(color, fill_alphas[0.80]),
        to_rgba(color, fill_alphas[0.50]),
    ]

    # Non-overlapping filled percentile bands
    ax.contourf(
        x_centers,
        magnitude_centers,
        histogram.T,
        levels=fill_levels,
        colors=fill_colors,
        antialiased=True,
        zorder=1,
    )

    # Corresponding contour boundaries
    for fraction in reversed(fractions):
        ax.contour(
            x_centers,
            magnitude_centers,
            histogram.T,
            levels=[thresholds[fraction]],
            colors=[color],
            linewidths=1.5,
            alpha=line_alphas[fraction],
            zorder=3,
        )


with plt.rc_context({
    "font.size": 10,
    "axes.labelsize": 11,
    "axes.linewidth": 0.8,
    "legend.fontsize": 8,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "xtick.major.size": 5,
    "ytick.major.size": 5,
    "xtick.minor.size": 3,
    "ytick.minor.size": 3,
}):
    fig, ax = plt.subplots(
        figsize=(5.2, 4.2),
        constrained_layout=True,
    )

    draw_filled_density(ax, hist_all, colors["all"])
    draw_filled_density(ax, hist_cut, colors["cut"])

    ax.axhline(
        mag_cut,
        color="grey",
        linestyle="--",
        linewidth=2,
        zorder=5,
    )

    ax.set_xscale("log")
    ax.set_xlabel(r"Stellar mass $[M_\odot]$")
    ax.set_ylabel(r"$M_g$")

    ax.set_xlim(
        10**logmass_edges[0],
        10**logmass_edges[-1],
    )
    ax.set_ylim(
        magnitude_edges[0],
        magnitude_edges[-1],
    )
    ax.invert_yaxis()

    ax.xaxis.set_minor_locator(
        LogLocator(
            base=10,
            subs=(0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9),
        )
    )
    ax.xaxis.set_minor_formatter(NullFormatter())

    ax.grid(color="0.88", linewidth=0.6)
    ax.set_axisbelow(True)

    legend_handles = [
        Line2D(
            [0], [0],
            color=colors["all"],
            linewidth=2,
            label="Without Stellar Mass Removal",
        ),
        Line2D(
            [0], [0],
            color=colors["cut"],
            linewidth=2,
            label="With Stellar Mass Removal",
        ),
        Line2D(
            [0], [0],
            color="grey",
            linestyle="--",
            linewidth=2,
            label=rf"$M_g={mag_cut:g}$",
        ),
    ]

    ax.legend(
        handles=legend_handles,
        frameon=False,
        loc="lower right",
        ncol=1,
    )

    fig.savefig("stellar_mass_magnitude.png", dpi=300, bbox_inches="tight")

    plt.show()

# Figure 3: X-ray panels

### Import data

In [ ]:
bahamas_path = '/projects/mccleary_group/habjan.e/TNG/Data'

cluster = "CDMb_GrNm_001"
b_data = np.load(bahamas_path + f"/CDMb/GrNm_001.npz")
xray_data_path = '/home/habjan.e/TNG/Sandbox_notebooks/mock_xray_maps/'

b_cop = b_data['sub_pos'][np.max(b_data['sub_massTotal'][:, 4]) == b_data['sub_massTotal'][:, 4], :]
boxsize = 400
difpos = np.subtract(b_data['gas_pos'], b_cop)
gaspos = (difpos + 0.5 * boxsize) % boxsize - 0.5 * boxsize

area       = (1000.0, "cm**2")

### Make plot

In [ ]:
# ------------------------------------------------------------------------------------
# Three-panel figure. All panels are in physical Mpc about the mosaic X-ray centre;
# +x is west (RA decreasing), +y is north, so each panel is in the usual sky orientation.
#   (a) ideal projected photons            -> photon surface brightness
#   (b) single 50 ks ACIS-I pointing       -> count-rate surface brightness (instrumental)
#   (c) 3x3 ACIS-I mosaic, exposure-corrected -> photon surface brightness
# Panels (a) and (c) carry the same units and agree to a few tens of per cent in the core;
# they differ because the mosaic's exposure map is evaluated at expmap_energy = 1 keV
# while the counts span 0.5-2 keV, and because the mosaic still contains the backgrounds.
# -----------------------------------------------------------------------------------

CMAP   = plt.get_cmap("afmhot").copy()
CMAP.set_bad(CMAP(0.0))          # unexposed pixels blend into the background
PCT    = 99.99                   # colour-bar top percentile, as in show()
A_samp = float(area[0])          # pyXSIM sampling area in cm^2 (not an effective area)


def _panel(img, header, norm, smooth):
    """Surface brightness per arcsec^2, plus the imshow extent in Mpc about the centre."""
    w_img  = wcs.WCS(header)
    pix_as = np.abs(w_img.wcs.cdelt[0]) * 3600.0                 # arcsec / pixel
    sb     = gaussian_filter(np.nan_to_num(img), smooth) / norm / pix_as ** 2
    x0, y0 = (float(v) for v in w_img.world_to_pixel(xray_center))
    s      = pix_as * Mpc_per_arcsec                             # Mpc / pixel
    ny, nx = img.shape
    ext    = [(-0.5 - x0) * s, (nx - 0.5 - x0) * s,
              (-0.5 - y0) * s, (ny - 0.5 - y0) * s]
    return sb, ext


def _lims(mask, ext, pad=0):
    """Axis limits enclosing the exposed pixels, so ACIS-I's 20' footprint is not
    swamped by the blank sky its 40'-wide image array carries around it."""
    ny, nx = mask.shape
    ix = np.flatnonzero(mask.any(axis=0))
    iy = np.flatnonzero(mask.any(axis=1))
    fx = lambda i: ext[0] + (i + 0.5) * (ext[1] - ext[0]) / nx
    fy = lambda j: ext[2] + (j + 0.5) * (ext[3] - ext[2]) / ny
    dx, dy = pad * (ext[1] - ext[0]), pad * (ext[3] - ext[2])
    return (fx(ix[0]) - dx, fx(ix[-1]) + dx), (fy(iy[0]) - dy, fy(iy[-1]) + dy)


# --- (a) ideal map: counts / (sampling area * photon-list exposure * pixel solid angle)
img_id = fits.getdata(xray_data_path + f"{cluster}_img_ideal.fits")
h_id   = fits.getheader(xray_data_path + f"{cluster}_img_ideal.fits")
sb_id, ext_id = _panel(img_id, h_id, A_samp * h_id["EXPOSURE"], 1.0)

# --- (b) single ACIS-I pointing: counts / (exposure * pixel solid angle), no exposure map,
#         so this is a raw instrumental count rate including all three backgrounds
img_ci = fits.getdata(xray_data_path + f"{cluster}_acisi_img.fits", "IMAGE")
h_ci   = fits.getheader(xray_data_path + f"{cluster}_acisi_img.fits", "IMAGE")
sb_ci, ext_ci = _panel(img_ci, h_ci, h_ci["EXPOSURE"], 2.0)
on_chip = gaussian_filter(img_ci, 2.0) > 0.0

# --- (c) mosaic: soxs writes .flux = counts / expmap with the exposure map normalised to
#         cm^2, so dividing by the per-pointing exposure gives photons s^-1 cm^-2
t_mos = fits.getheader(xray_data_path + f"{cluster}_mosaic_0_img.fits", "IMAGE")["EXPOSURE"]
sb_mo, ext_mo = _panel(flux_mos, hdr_mos, t_mos, 3.0)
sb_mo[~cov_mos] = np.nan

R200_Mpc = R200 * a / hub
ext_id = [-5, 5, -5, 5]
lim_id = _lims(np.ones_like(sb_id, dtype=bool), ext_id)   # whole loaded region
lim_ci = _lims(on_chip, ext_ci)                           # the four ACIS-I chips
lim_mo = _lims(cov_mos, ext_mo)                           # the mosaic footprint

# (image, extent, axis limits, panel label, colour-bar label, colour-bar dynamic range).
# The dynamic range is 1e3 as in show(), except for (b): a single pointing is filled edge
# to edge by cluster plus background, so 1e3 puts the whole chip near the top of the bar.
panels = [
    (sb_id, ext_id, lim_id, "(a) No instrumental effects",
     r"$\rm counts\, \, s^{-1}\, cm^{-2}\, arcsec^{-2}$", 1.0e3),
    (sb_ci, ext_ci, lim_ci, f"(b) ACIS-I, {h_ci['EXPOSURE']/1e3:.0f} ks",
     r"$\rm counts\, \, s^{-1}\, arcsec^{-2}$", 5.0e1),
    (sb_mo, ext_mo, lim_mo,
     f"(c) {n_side}$\\times${n_side} ACIS-I mosaic, {t_mos/1e3:.0f} ks each",
     r"$\rm counts\, \, s^{-1}\, cm^{-2}\, arcsec^{-2}$", 1.0e3),
]


def _cb_ticks(vmin, vmax):
    """Decade ticks, thinned out with 3x steps when the range is under two decades."""
    lo, hi = np.log10(vmin), np.log10(vmax)
    t = 10.0 ** np.arange(np.ceil(lo), np.floor(hi) + 1)
    if len(t) < 3:
        t = np.sort(np.concatenate([t, 3.0 * 10.0 ** np.arange(np.floor(lo), hi)]))
    t = t[(t >= vmin) & (t <= vmax)]
    lab = [(r"$10^{%d}$" % round(np.log10(v))) if abs(np.log10(v) - round(np.log10(v))) < 1e-6
           else (r"$3\times10^{%d}$" % np.floor(np.log10(v))) for v in t]
    return t, lab


fig, axes = plt.subplots(1, 3, figsize=(15.0, 5.5))
for ax, (sb, ext, lim, label, cb_label, dyn) in zip(axes, panels):
    vmax = np.nanpercentile(sb, PCT)
    im = ax.imshow(sb, origin="lower", extent=ext, cmap=CMAP, interpolation="nearest",
                   norm=LogNorm(vmin=vmax / dyn, vmax=vmax))
    ax.set_xlim(*lim[0])
    ax.set_ylim(*lim[1])
    ax.plot(0.0, 0.0, "+", color="#41d3ff", ms=10, mew=1.5, zorder=5)   # X-ray centre
    ax.set_xlabel("X-coordinate [Mpc]", fontsize=14, fontweight='semibold')
    ax.set_ylabel("Y-coordinate [Mpc]", fontsize=14, fontweight='semibold')
    ax.tick_params(which="both", direction="in", color="white", labelsize=11,
                   top=True, right=True)
    ax.minorticks_on()
    ax.text(0.035, 0.965, label, transform=ax.transAxes, color="white",
            fontsize=12.5, va="top", ha="left")
    cax = make_axes_locatable(ax).append_axes("top", size="4.5%", pad=0.07)
    cb = fig.colorbar(im, cax=cax, orientation="horizontal")
    cb.set_label(cb_label, fontsize=11.5, labelpad=8)
    ticks, tlabels = _cb_ticks(vmax / dyn, vmax)
    cb.set_ticks(ticks)
    cax.set_xticklabels(tlabels)
    cax.xaxis.set_ticks_position("top")
    cax.xaxis.set_label_position("top")
    cax.tick_params(labelsize=10)

# R200 on the ideal panel, and the footprint of each narrower panel on the wider ones,
# to make the nesting of scales explicit
axes[0].add_patch(Circle((0, 0), R200_Mpc, fill=False, ec="#41d3ff", lw=0.9, ls="-",
                         alpha=0.9))
axes[0].text(0.0, R200_Mpc, r"$R_{200}$", color="#41d3ff", fontsize=11,
             ha="center", va="bottom")
for ax, lim in [(axes[0], lim_mo), (axes[0], lim_ci), (axes[2], lim_ci)]:
    ax.add_patch(Rectangle((lim[0][0], lim[1][0]), lim[0][1] - lim[0][0],
                           lim[1][1] - lim[1][0], fill=False, ec="#41d3ff", lw=1.5,
                           ls="--", alpha=0.75))

mexp = int(np.floor(np.log10(M200)))
#fig.suptitle(f"{cluster}:  0.5$-$2 keV,  $z = {redshift}$,  "
 #            f"$M_{{200}} = {M200/10**mexp:.2f}\\times10^{{{mexp}}}\\,M_\\odot$,  "
  #           f"$R_{{200}} = {R200_Mpc:.2f}$ Mpc", fontsize=13.5, y=1.02)
fig.tight_layout()
#fig.savefig(f"{cluster}_xray_3panel.pdf", bbox_inches="tight")
fig.savefig(f"{cluster}_xray_3panel.png", dpi=300, bbox_inches="tight")
plt.show()

# Figure 4: Shape catalog plot. Redshift distribution, 2D source catalog plot, shape noise

# Figure 5: Model plot

# Figure 6: Mass-to-Mass comparison for different ablations

### Import shape, dynamics, xray data

In [ ]:
val_path = "/projects/mccleary_group/habjan.e/TNG/Data/shape_dynamics/shape_dynamics_val.h5"
data_dict = preload_hdf5_to_memory(val_path)

### Import samples

In [ ]:
data_dirc = "/projects/mccleary_group/habjan.e/TNG/Data/conditional_diffusion_data/posterior_sampling"
folder = Path(data_dirc)

true_cube_arr, sample_cube_arr, z_lens_arr = [], [], []

for path in folder.glob(f"*{suffix}.npz"):

    data = np.load(data_dirc + '/' + path.name)

    true_cube_arr.append(data['true_cube'])
    sample_cube_arr.append(data['sampled_cubes'])
    z_lens_arr.append(data['z_lens'])

true_cube_arr = np.array(true_cube_arr)
sample_cube_arr = np.array(sample_cube_arr)
z_lens_arr = np.array(z_lens_arr)

true_cube_arr.shape, sample_cube_arr.shape, z_lens_arr.shape

### Mass function

In [ ]:
def cube_to_mass_msun(cube_norm, metadata):
    """
    cube_norm: (Z,Y,X) normalized/log-standardized cube from your dataset or samples
    metadata: HDF5 attrs dict from preload_hdf5_to_memory(... )["metadata"]

    returns:
        enclosed mass inside the cube in Msun
    """
    cube_mean = float(metadata["cube_log10_mean"])
    cube_std = float(metadata["cube_log10_std"])
    floor_value = float(metadata["floor_value"])
    fov_mpc = float(metadata["map_fov_mpc"])
    N = int(metadata["cube_resolution"])

    # invert normalization
    rho = np.zeros_like(cube_norm, dtype=np.float64)
    mask = cube_norm > floor_value + 1e-6
    rho[mask] = 10.0 ** (cube_norm[mask] * cube_std + cube_mean)  # Msun / Mpc^3

    # voxel volume
    voxel_size = (2.0 * fov_mpc) / N
    voxel_vol = voxel_size ** 3  # Mpc^3

    mass = np.sum(rho) * voxel_vol
    return mass

### Make plot

In [ ]:
# Expected shapes:
# true_cube_arr:   (50, 16, 16, 16)
# sample_cube_arr: (50, 200, 16, 16, 16)
# z_lens_arr:      (50,)

n_cubes, n_samples = sample_cube_arr.shape[:2]

assert true_cube_arr.shape[0] == n_cubes
assert true_cube_arr.shape[1:] == sample_cube_arr.shape[2:]
assert len(z_lens_arr) == n_cubes


# Calculate log10 masses
true_masses = np.empty(n_cubes)
sample_masses = np.empty((n_cubes, n_samples))

for i in range(n_cubes):
    true_mass = cube_to_mass_msun(
        true_cube_arr[i],
        metadata,
    )

    if true_mass <= 0:
        raise ValueError(f"True mass for cube {i} is not positive.")

    true_masses[i] = np.log10(true_mass)

    for j in range(n_samples):
        sampled_mass = cube_to_mass_msun(
            sample_cube_arr[i, j],
            metadata,
        )

        if sampled_mass <= 0:
            raise ValueError(
                f"Sampled mass for cube {i}, sample {j} is not positive."
            )

        sample_masses[i, j] = np.log10(sampled_mass)


# Statistics are calculated after transforming to log10 space
sample_median = np.percentile(sample_masses, 50, axis=1)
sample_p16 = np.percentile(sample_masses, 16, axis=1)
sample_p84 = np.percentile(sample_masses, 84, axis=1)

lower_error = sample_median - sample_p16
upper_error = sample_p84 - sample_median


# Use identical x and y limits
limit_min = min(
    np.min(true_masses),
    np.min(sample_p16),
)

limit_max = max(
    np.max(true_masses),
    np.max(sample_p84),
)

padding = 0.05 * (limit_max - limit_min)
plot_min = limit_min - padding
plot_max = limit_max + padding


# Map lens redshift to colors
z_lens = np.asarray(z_lens_arr)
cmap = plt.colormaps["viridis"]
norm = Normalize(
    vmin=np.nanmin(z_lens),
    vmax=np.nanmax(z_lens),
)
point_colors = cmap(norm(z_lens))


fig, ax = plt.subplots(figsize=(7.5, 6.5))

# Draw individually so each error bar matches its point color
for i in range(n_cubes):
    ax.errorbar(
        true_masses[i],
        sample_median[i],
        yerr=[
            [lower_error[i]],
            [upper_error[i]],
        ],
        fmt="none",
        ecolor=point_colors[i],
        elinewidth=1.5,
        capsize=3,
        alpha=0.8,
        zorder=1,
    )

points = ax.scatter(
    true_masses,
    sample_median,
    c=z_lens,
    cmap=cmap,
    norm=norm,
    s=55,
    edgecolor="black",
    linewidth=0.5,
    zorder=2,
)

# Black 1-to-1 line
ax.plot(
    [plot_min, plot_max],
    [plot_min, plot_max],
    color="black",
    linestyle="-",
    linewidth=1.5,
    label="1:1",
    zorder=0,
)

ax.set_xlim(plot_min, plot_max)
ax.set_ylim(plot_min, plot_max)
ax.set_aspect("equal", adjustable="box")

ax.set_xlabel(
    r"True enclosed mass, $\log_{10}(M_{\mathrm{true}}/M_\odot)$"
)
ax.set_ylabel(
    r"Median sampled mass, $\log_{10}(M_{\mathrm{sample}}/M_\odot)$"
)
ax.grid(alpha=0.2)
ax.legend()

colorbar = fig.colorbar(points, ax=ax)
colorbar.set_label(r"$z_{\mathrm{lens}}$")

plt.tight_layout()
fig.savefig(f"mass_to_mass.png", dpi=300, bbox_inches="tight")
plt.show()

# Figure 7: Axes length comparison for different ablations

### Shapes function

In [ ]:
def cube_to_axis_lengths_mpc(cube_norm, metadata):
    """
    cube_norm: (Z,Y,X) normalized/log-standardized cube from your dataset or samples
    metadata: HDF5 attrs dict from preload_hdf5_to_memory(... )["metadata"]

    returns:
        a, b, c : characteristic mass-weighted axis lengths in Mpc,
                  ordered from largest to smallest
    """
    cube_mean = float(metadata["cube_log10_mean"])
    cube_std = float(metadata["cube_log10_std"])
    floor_value = float(metadata["floor_value"])
    fov_mpc = float(metadata["map_fov_mpc"])
    N = int(metadata["cube_resolution"])

    # invert normalization: rho in Msun / Mpc^3
    rho = np.zeros_like(cube_norm, dtype=np.float64)
    mask = cube_norm > floor_value + 1e-6
    rho[mask] = 10.0 ** (cube_norm[mask] * cube_std + cube_mean)

    # voxel geometry
    voxel_size = (2.0 * fov_mpc) / N
    voxel_vol = voxel_size ** 3

    # voxel masses in Msun
    mass = rho * voxel_vol
    total_mass = np.sum(mass)

    if total_mass <= 0:
        return np.nan, np.nan, np.nan

    # voxel-center coordinates in Mpc
    coords_1d = (np.arange(N, dtype=np.float64) + 0.5) * voxel_size - fov_mpc
    z, y, x = np.meshgrid(coords_1d, coords_1d, coords_1d, indexing="ij")

    # center of mass
    x_com = np.sum(mass * x) / total_mass
    y_com = np.sum(mass * y) / total_mass
    z_com = np.sum(mass * z) / total_mass

    dx = x - x_com
    dy = y - y_com
    dz = z - z_com

    # mass-weighted shape tensor
    S_xx = np.sum(mass * dx * dx) / total_mass
    S_yy = np.sum(mass * dy * dy) / total_mass
    S_zz = np.sum(mass * dz * dz) / total_mass
    S_xy = np.sum(mass * dx * dy) / total_mass
    S_xz = np.sum(mass * dx * dz) / total_mass
    S_yz = np.sum(mass * dy * dz) / total_mass

    S = np.array([
        [S_xx, S_xy, S_xz],
        [S_xy, S_yy, S_yz],
        [S_xz, S_yz, S_zz]
    ], dtype=np.float64)

    # diagonalize and sort largest -> smallest
    evals = np.linalg.eigvalsh(S)
    evals = np.sort(evals)[::-1]

    # characteristic axis lengths
    a, b, c = np.sqrt(np.clip(evals, 0.0, None))

    return float(a), float(b), float(c)

### Make plot

In [ ]:
    fig.savefig(
        "/home/habjan.e/TNG/cluster_deprojection/figures/"
        "cube_shapes.png",
        dpi=300,
        bbox_inches="tight",
    )

# Figure 8: Power Spectrum plot

### Power spectrum function

In [ ]:
def cube_to_mass_power_spectrum(
    cube_norm,
    metadata,
    lower_scale_limit=1.0,
    upper_scale_limit=100.0,
    n_bins=25,
):
    """
    Calculate the spherically averaged 3D mass-density power spectrum.

    Parameters
    ----------
    cube_norm : ndarray, shape (Z, Y, X)
        Normalized/log-standardized mass-density cube.

    metadata : dict
        HDF5 metadata containing:
            cube_log10_mean
            cube_log10_std
            floor_value
            map_fov_mpc
            cube_resolution

    lower_scale_limit : float, optional
        Smallest requested physical scale in Mpc. Default is 1 Mpc.

    upper_scale_limit : float, optional
        Largest requested physical scale in Mpc. Default is 100 Mpc.

    n_bins : int, optional
        Number of logarithmic scale bins. Default is 25.

    Returns
    -------
    result : dict
        Dictionary containing:

        scale_mpc
            Effective physical scale of each bin, where scale = 1/k.

        k_mpc_inv
            Effective spatial frequency in Mpc^-1.

        power
            Spherically averaged density power spectrum.
            Units: Msun^2 / Mpc^3.

        power_error
            Gaussian mode-counting uncertainty:
            P(k) * sqrt(2 / N_modes).

        n_modes
            Number of Fourier modes in each bin.

        scale_edges_mpc
            Scale-bin edges actually used.

        scale_bounds_used_mpc
            Effective lower and upper scale limits after restricting
            the calculation to scales supported by the cube.
    """
    cube_norm = np.asarray(cube_norm, dtype=np.float64)

    if cube_norm.ndim != 3:
        raise ValueError(
            f"cube_norm must have shape (Z, Y, X); got {cube_norm.shape}"
        )

    if lower_scale_limit <= 0:
        raise ValueError("lower_scale_limit must be positive.")

    if upper_scale_limit <= lower_scale_limit:
        raise ValueError(
            "upper_scale_limit must be greater than lower_scale_limit."
        )

    if n_bins < 1:
        raise ValueError("n_bins must be at least 1.")

    # ---------------------------------------------------------------
    # 1. Convert the normalized cube to physical density
    # ---------------------------------------------------------------
    cube_mean = float(metadata["cube_log10_mean"])
    cube_std = float(metadata["cube_log10_std"])
    floor_value = float(metadata["floor_value"])
    fov_mpc = float(metadata["map_fov_mpc"])
    cube_resolution = int(metadata["cube_resolution"])

    rho = np.zeros_like(cube_norm, dtype=np.float64)

    physical_mask = (
        np.isfinite(cube_norm)
        & (cube_norm > floor_value + 1.0e-6)
    )

    rho[physical_mask] = 10.0 ** (
        cube_norm[physical_mask] * cube_std + cube_mean
    )
    # rho has units Msun / Mpc^3

    # ---------------------------------------------------------------
    # 2. Set up the real-space voxel and box dimensions
    # ---------------------------------------------------------------
    voxel_size_mpc = (2.0 * fov_mpc) / cube_resolution

    nz, ny, nx = rho.shape

    box_lengths_mpc = voxel_size_mpc * np.array(
        [nz, ny, nx],
        dtype=float,
    )
    box_volume_mpc3 = np.prod(box_lengths_mpc)

    # ---------------------------------------------------------------
    # 3. Calculate the volume-normalized 3D power spectrum
    # ---------------------------------------------------------------
    rho_fft = np.fft.fftn(rho) / rho.size

    # Conventional volume-normalized density power:
    #
    # P(k) = V |FFT(rho) / N_voxels|^2
    #
    power_3d = box_volume_mpc3 * np.abs(rho_fft) ** 2

    # ---------------------------------------------------------------
    # 4. Build the Fourier-frequency grid
    #
    # The cube ordering is (Z, Y, X), so the frequency arrays follow
    # exactly that same ordering.
    # ---------------------------------------------------------------
    kz = np.fft.fftfreq(nz, d=voxel_size_mpc)
    ky = np.fft.fftfreq(ny, d=voxel_size_mpc)
    kx = np.fft.fftfreq(nx, d=voxel_size_mpc)

    k_magnitude = np.sqrt(
        kz[:, None, None] ** 2
        + ky[None, :, None] ** 2
        + kx[None, None, :] ** 2
    )

    # Use the inscribed Fourier sphere, avoiding anisotropically
    # sampled corner modes above the one-dimensional Nyquist limit.
    k_nyquist = 1.0 / (2.0 * voxel_size_mpc)

    nonzero_k = k_magnitude[
        (k_magnitude > 0)
        & (k_magnitude <= k_nyquist)
    ]

    if nonzero_k.size == 0:
        raise ValueError("The cube contains no usable nonzero Fourier modes.")

    # Scales that can actually be measured from this cube.
    smallest_supported_scale = 1.0 / np.max(nonzero_k)
    largest_supported_scale = 1.0 / np.min(nonzero_k)

    scale_min = max(
        float(lower_scale_limit),
        smallest_supported_scale,
    )
    scale_max = min(
        float(upper_scale_limit),
        largest_supported_scale,
    )

    if scale_min >= scale_max:
        raise ValueError(
            "The requested scale interval does not overlap the scales "
            "supported by the cube. "
            f"Requested: [{lower_scale_limit}, "
            f"{upper_scale_limit}] Mpc; "
            f"supported: [{smallest_supported_scale:.4g}, "
            f"{largest_supported_scale:.4g}] Mpc."
        )

    # ---------------------------------------------------------------
    # 5. Construct logarithmic bins
    # ---------------------------------------------------------------
    scale_edges = np.logspace(
        np.log10(scale_min),
        np.log10(scale_max),
        n_bins + 1,
    )

    # scale = 1/k. Reversing produces increasing k-bin edges.
    k_edges = (1.0 / scale_edges)[::-1]

    valid_modes = (
        np.isfinite(power_3d)
        & (k_magnitude > 0)
        & (k_magnitude <= k_nyquist)
        & (k_magnitude >= k_edges[0])
        & (k_magnitude <= k_edges[-1])
    )

    k_values = k_magnitude[valid_modes]
    power_values = power_3d[valid_modes]

    # ---------------------------------------------------------------
    # 6. Spherically average the power in each Fourier shell
    # ---------------------------------------------------------------
    n_modes, _ = np.histogram(
        k_values,
        bins=k_edges,
    )

    power_sum, _ = np.histogram(
        k_values,
        bins=k_edges,
        weights=power_values,
    )

    k_sum, _ = np.histogram(
        k_values,
        bins=k_edges,
        weights=k_values,
    )

    power = np.full(n_bins, np.nan, dtype=float)
    power_error = np.full(n_bins, np.nan, dtype=float)
    k_effective = np.full(n_bins, np.nan, dtype=float)

    populated = n_modes > 0

    power[populated] = (
        power_sum[populated] / n_modes[populated]
    )

    k_effective[populated] = (
        k_sum[populated] / n_modes[populated]
    )

    power_error[populated] = (
        power[populated]
        * np.sqrt(2.0 / n_modes[populated])
    )

    scale_effective = 1.0 / k_effective

    # The k bins are ascending, so their corresponding scales are
    # descending. Sort everything into ascending physical scale.
    order = np.argsort(scale_effective[populated])

    return {
        "scale_mpc": scale_effective[populated][order],
        "k_mpc_inv": k_effective[populated][order],
        "power": power[populated][order],
        "power_error": power_error[populated][order],
        "n_modes": n_modes[populated][order],
        "scale_edges_mpc": scale_edges,
        "scale_bounds_used_mpc": (scale_min, scale_max),
    }

### Create power spectra

In [ ]:
low_scale_lim = 1
upper_scale_lim = 10

true_spectrum = cube_to_mass_power_spectrum(
    data_true_cube,
    metadata,
    lower_scale_limit=low_scale_lim,
    upper_scale_limit=upper_scale_lim,
)

scale_mpc = true_spectrum["scale_mpc"]
true_power = true_spectrum["power"]

sample_spectra = [
    cube_to_mass_power_spectrum(
        data_sampled_cubes[i],
        metadata,
        lower_scale_limit=low_scale_lim,
        upper_scale_limit=upper_scale_lim,
    )
    for i in range(data_sampled_cubes.shape[0])
]

sample_power = np.stack([
    result["power"]
    for result in sample_spectra
])

power_mean = np.nanmean(sample_power, axis=0)
power_std = np.nanstd(sample_power, axis=0)

power_p16, power_median, power_p84 = np.nanpercentile(
    sample_power,
    [16, 50, 84],
    axis=0,
)

### Make plot

In [ ]:
# Extract the true spectrum
scale_mpc = np.asarray(true_spectrum["scale_mpc"])
true_power = np.asarray(true_spectrum["power"])

# sample_power has shape (n_samples, n_scale_bins)
power_mean = np.nanmean(sample_power, axis=0)
power_std = np.nanstd(sample_power, axis=0, ddof=1)

power_lower = power_mean - power_std
power_upper = power_mean + power_std

# Valid values for logarithmic plotting
valid_true = (
    np.isfinite(scale_mpc)
    & np.isfinite(true_power)
    & (scale_mpc > 0)
    & (true_power > 0)
)

valid_mean = (
    np.isfinite(scale_mpc)
    & np.isfinite(power_mean)
    & (scale_mpc > 0)
    & (power_mean > 0)
)

# A log axis cannot display a lower uncertainty bound <= 0
valid_band = (
    valid_mean
    & np.isfinite(power_lower)
    & np.isfinite(power_upper)
    & (power_lower > 0)
    & (power_upper > 0)
)

with plt.rc_context({
    "font.family": "serif",
    "mathtext.fontset": "stix",
    "font.size": 11,
    "axes.labelsize": 13,
    "axes.linewidth": 1.1,
    "legend.fontsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "xtick.major.size": 6,
    "ytick.major.size": 6,
    "xtick.minor.size": 3,
    "ytick.minor.size": 3,
}):
    fig, ax = plt.subplots(
        figsize=(6.5, 4.8),
        constrained_layout=True,
    )

    # Posterior mean
    ax.plot(scale_mpc[valid_mean], power_mean[valid_mean],
    color="#4477AA", linewidth=2.2, label="Posterior mean", zorder=3, )

    # Posterior mean ± one standard deviation
    ax.fill_between(scale_mpc, power_lower, power_upper, where=valid_band,
        interpolate=True, color="#4477AA", alpha=0.25, linewidth=0, label=r"Posterior mean $\pm 1\sigma$", zorder=1,)

    # True power spectrum
    ax.plot(scale_mpc[valid_true], true_power[valid_true],
        color="#CC3311", linewidth=2.2, linestyle="--", label="True", zorder=4,)

    ax.set_xscale("log")
    ax.set_yscale("log")

    ax.set_xlabel(r"$1/k\;[\mathrm{Mpc}]$", fontsize=14, fontweight='semibold')
    ax.set_ylabel(r"$P(k)\, \, \;[M_{\odot}^{2}\,\mathrm{Mpc}^{-3}]$", fontsize=14, fontweight='semibold')

    ax.grid(which="major", color="0.85", linestyle="-", linewidth=0.7, alpha=0.7,)

    ax.legend(loc="best", frameon=False, handlelength=2.8,)

    fig.savefig("/home/habjan.e/TNG/cluster_deprojection/figures/mass_density_power_spectrum.png", dpi=500, bbox_inches="tight",)

    plt.show()

# Figure 9: Mass, axes length sample plot with prior

### Import data for a sample

In [ ]:
suffix = "_shape_dyn_v1" #"_16cube_16img_v11"

sample_num = '0071'
sample_num_int = int(sample_num)

sample_base_path = '/projects/mccleary_group/habjan.e/TNG/Data/conditional_diffusion_data/posterior_sampling/'
sample_path = sample_base_path + f'posterior_example_{sample_num}{suffix}.npz'
data = np.load(sample_path)
data_im, data_true_cube, data_sampled_cubes, data_z_lens, data_sim_idx, data_sim = data['conditioning_images'], data['true_cube'], data['sampled_cubes'], data['z_lens'], data['cluster_index'], data['simulation']

print(data_sim_idx, data_sim, data_z_lens)

idx = sample_num_int

### Mass arrays

In [ ]:
metadata = data_dict["metadata"]

true_masses = np.log10(cube_to_mass_msun(data_true_cube[:, :, :], metadata))
sample_masses = np.log10(np.array([cube_to_mass_msun(data_sampled_cubes[i, :, :, :], metadata) for i in range(data_sampled_cubes.shape[0])]))

### Shape arrays

In [ ]:
true_shapes = np.array([cube_to_axis_lengths_mpc(data_true_cube[:, :, :], metadata)]).flatten()
sample_shapes = np.array([cube_to_axis_lengths_mpc(data_sampled_cubes[i, :, :, :], metadata) for i in range(data_sampled_cubes.shape[0])])

### Make the plot

In [ ]:
true_mass = float(np.asarray(true_masses))
sample_masses_plot = np.asarray(sample_masses).ravel()

true_shapes_plot = np.asarray(true_shapes).ravel()
sample_shapes_plot = np.asarray(sample_shapes)

shape_axis_names = ("a", "b", "c")

samples_to_plot = [sample_masses_plot,sample_shapes_plot[:, 0],sample_shapes_plot[:, 1],sample_shapes_plot[:, 2],]
true_values = [true_mass,true_shapes_plot[0],true_shapes_plot[1],true_shapes_plot[2],]

xlabels = [
    r"$\rm Cube$ $\rm \log_{10}(M/M_{\odot})$",
    rf"${shape_axis_names[0]}\;[\mathrm{{Mpc}}]$",
    rf"${shape_axis_names[1]}\;[\mathrm{{Mpc}}]$",
    rf"${shape_axis_names[2]}\;[\mathrm{{Mpc}}]$",
]

hist_color = "#2A6F97"
true_color = "#CC3311"

with plt.rc_context({
    "font.size": 11,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "axes.linewidth": 1.0,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "savefig.dpi": 500,
}):
    fig, axes = plt.subplots(
        1,
        4,
        figsize=(14, 3.4),
        constrained_layout=True,
    )

    for i, (ax, samples, true_value, xlabel) in enumerate(
        zip(axes, samples_to_plot, true_values, xlabels)
    ):
        # Remove NaN and infinite posterior samples
        samples = np.asarray(samples)
        samples = samples[np.isfinite(samples)]

        ax.hist(
            samples,
            bins="auto",
            density=True,
            color=hist_color,
            alpha=0.75,
            edgecolor="white",
            linewidth=0.7,
            label="Posterior samples",
        )

        ax.axvline(
            true_value,
            color=true_color,
            linestyle="--",
            linewidth=2.0,
            zorder=5,
            label="True value",
        )

        ax.set_xlabel(xlabel)
        ax.set_ylabel("Posterior density" if i == 0 else "")
        ax.tick_params(which="both", width=1.0)

    # One legend for the complete figure
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="outside upper center",
        ncols=2,
        frameon=False,
    )

    fig.savefig(
        "/home/habjan.e/TNG/cluster_deprojection/figures/"
        "mass_and_shape_posteriors.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()

# Figure 10: xy- and xz-plane distribution of a selected cluster

In [ ]:
cube_mean = data_dict["metadata"]["cube_log10_mean"]
cube_std  = data_dict["metadata"]["cube_log10_std"]

# Convert normalized log-density cubes back to linear density.
rho_true = 10 ** (data_true_cube * cube_std + cube_mean)
rho_samples = 10 ** (data_sampled_cubes * cube_std + cube_mean)

# data_true_cube shape:     (z, y, x)
# data_sampled_cubes shape: (samples, z, y, x)

with np.errstate(divide="ignore", invalid="ignore"):
    # XY projections: sum along z.
    true_xy = np.log10(np.sum(rho_true, axis=0))
    sampled_xy = np.log10(np.sum(rho_samples, axis=1))

    # XZ projections: sum along y.
    true_xz = np.log10(np.sum(rho_true, axis=1))
    sampled_xz = np.log10(np.sum(rho_samples, axis=2))

mean_xy = np.nanmean(sampled_xy, axis=0)
std_xy  = np.nanstd(sampled_xy, axis=0)

mean_xz = np.nanmean(sampled_xz, axis=0)
std_xz  = np.nanstd(sampled_xz, axis=0)

images = [
    [true_xy, mean_xy, std_xy],
    [true_xz, mean_xz, std_xz],
]

titles = [
    "True projection",
    "Mean sampled projection",
    "Sample standard deviation",
]

ylabels = ["Y-coordinate", "Z-coordinate"]
cmap = "cubehelix"

fig, axes = plt.subplots(
    nrows=2,
    ncols=3,
    figsize=(15, 10),
    constrained_layout=False,
)

fig.subplots_adjust(
    wspace=0.50,
    hspace=0.0,
)

# Use consistent scales for directly comparable panels.
mass_images = [true_xy, mean_xy, true_xz, mean_xz]
mass_vmin = min(np.nanmin(image) for image in mass_images)
mass_vmax = max(np.nanmax(image) for image in mass_images)
std_vmax = max(np.nanmax(std_xy), np.nanmax(std_xz))

for row in range(2):
    for col in range(3):
        ax = axes[row, col]

        if col < 2:
            im = ax.imshow(
                images[row][col],
                cmap=cmap,
                origin="upper",
                vmin=mass_vmin,
                vmax=mass_vmax,
            )
            colorbar_label = r"$\log_{10}(M/M_{\odot})$"
        else:
            im = ax.imshow(
                images[row][col],
                cmap=cmap,
                origin="upper",
                vmin=0,
                vmax=std_vmax,
            )
            colorbar_label = r"$\sigma_{\log_{10}(M/M_{\odot})}$"

        ax.set_xlabel(
            "X-coordinate",
            fontsize=14,
            fontweight="semibold",
        )
        ax.set_ylabel(
            ylabels[row],
            fontsize=14,
            fontweight="semibold",
        )

        if row == 0:
            ax.set_title(
                titles[col],
                fontsize=15,
                fontweight="semibold",
            )

        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
        cbar.set_label(
            colorbar_label,
            fontsize=14,
            fontweight="semibold",
        )


fig.savefig(
    "/home/habjan.e/TNG/cluster_deprojection/figures/"
    "xy_xz_mass_distribution.png",
    dpi=300,
    bbox_inches="tight",
    )
plt.show()